# 12. Figures and Table 1

Draws the figures of the paper from the saved results of notebooks 02–11 (run those first).
Panels were assembled into the multi-panel figures of the paper outside Python.

| Paper | Cell | Output |
|---|---|---|
| Figure 2-A | Final model performance | `outputs/3_final/plots/XGBoost_final_model.png` |
| Figure 2-B, Table 1 | Attitude variables among the top 10 | `outputs/4_updown/` |
| Figure 2-C, 2-D, 2-F, S4 | Reduced models | `outputs/4_parsimonious/plots/` |
| Figure 2-E | Models by variable category | `outputs/4_parsimonious/plots/XGBoost_comparison_category_models.png` |
| Figure 5 | Race and sex models | `outputs/7_final_race_sex/plots/` |
| Figure S1-A | Preliminary model | `outputs/1_preliminary/plots/XGBoost_performance_over_time.png` |
| Figure S1-B | see `robustness_missing_types/` | |
| Figure S2 | Preliminary logistic regression | `outputs/1_1_preliminary_logit/plots/` |
| Figure S3 | Feature selection criteria | `outputs/2_feature_selection/plots/` |

Figures 3 and 4 and Table S1 are produced by `07_descriptives.ipynb`.

Lines are mean AUROC over the 20 repetitions; shades are ±2 SD across repetitions (Figure 5: ±1 SD).

## Figure 2-A: final model

In [ ]:
### Performance by year

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Settings ---
# Result file
excel_file_path = 'outputs/3_final/3_FINAL_M3_Top40_partyid_1_analysis_results_weighted.xlsx'

# Output folder
output_directory = 'outputs/3_final/plots'

# Sheet
sheet_name = '2_Performance_Compare'

# --- Read data ---
try:
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
    print(f"Read sheet '{sheet_name}' of '{excel_file_path}'.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: '{excel_file_path}' not found.")
except ValueError:
    print(f"Error: sheet '{sheet_name}' not found in '{excel_file_path}'.")
except Exception as e:
    print(f"Error while reading the Excel file: {e}")
    df = None # skip the plotting below on error

# --- Prepare data and plot ---
if df is not None:
    # Check required columns
    required_cols = ['year', 'ballot', 'model', 'avg_score_mean', 'avg_score_std']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: required columns ({required_cols}) missing.")
    else:
        # 'year' to numeric
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        # Keep rows with a valid 'year'
        df.dropna(subset=['year'], inplace=True)
        df['year'] = df['year'].astype(int)

        # Create the output folder if needed
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)

        # Plot style
        plt.style.use('seaborn-v0_8-white')

        # One plot per model
        models_to_process = df['model'].unique()

        print("\nPlotting...")

        for model_name in models_to_process:
            model_data = df[df['model'] == model_name].copy()

            if model_data.empty:
                print(f"No data for model '{model_name}'; no plot.")
                continue

            plt.figure(figsize=(12, 9))
            ax = plt.gca()

            # Ballots in model_data
            ballots = model_data['ballot'].unique()

            # One color per ballot (seaborn deep palette)
            pal = sns.color_palette('deep')
            BALLOT_COLORS = {'a': pal[0], 'b': pal[1], 'c': pal[2]}
            palette = [BALLOT_COLORS[b] for b in ballots]

            for i, ballot_name in enumerate(ballots):
                # Data of one ballot, sorted by year
                ballot_data = model_data[model_data['ballot'] == ballot_name].sort_values('year')

                if ballot_data.empty:
                    continue

                x = ballot_data['year']
                y_mean = ballot_data['avg_score_mean']
                y_std = ballot_data['avg_score_std']

                # Mean score (avg_score_mean) as a line
                ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])

                # Standard deviation (avg_score_std) as a shaded band
                # fill_between fills the range of y for each x.
                # Shade: ±2 SD across the 20 repetitions
                ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)



            # Title and axis labels
            ax.set_title(f'Final Model Performance', fontsize=16, pad=20)
            ax.set_xlabel('Year', fontsize=12)
            ax.set_ylabel('Average Score (AUROC)', fontsize=12)
            ax.legend(title='Ballot')
            ax.set_ylim(0.5, 1)
            ax.set_xlim(1986, 2026)

            # No grid
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            plt.tight_layout()

            # Save
            filename = f'{model_name}_final_model.png'
            filepath = os.path.join(output_directory, filename)
            plt.savefig(filepath, dpi=300)
            print(f"Saved '{filepath}'.")

            plt.close()

        print("\nAll plots done.")

## Figure 2-B and Table 1: attitude variables among the 10 most important variables

SHAP importance of the final model (`05_final_model.ipynb`), averaged over the 20 runs; ranks within each ballot and year.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import textwrap
import matplotlib.patheffects as path_effects

# ===== Parameters =====
SNAPSHOT_YEARS = ['1988', '2006', '2024']   # GSS survey years
TARGET_MODEL = 'XGBoost'
TOP_N = 10

# ===== File paths =====
IMPORTANCE_FILE = 'outputs/3_final/3_FINAL_M3_Top40_partyid_1_analysis_all_importance_weighted.parquet'
METADATA_FILE = "inputs/top_40_features_m3.xlsx"
METADATA_SHEET = "XGBVARS"   # variables of the final XGBoost model

# ===== Display constants derived from the parameters =====
CLIPPED_RANK = TOP_N + 1                       # ranks outside the top N are clipped to this value
DISPLAY_RANK_SINK = TOP_N + 0.5                # y position of the 'N+' node
SHADE_MIDPOINT = (TOP_N + DISPLAY_RANK_SINK) / 2
YEAR_SUFFIX = "_".join(y[2:] for y in SNAPSHOT_YEARS)   # for file names (e.g. 88_06_24)
os.makedirs('outputs/4_updown', exist_ok=True)

# ===== Variable metadata (label, attitude) =====
df_meta_map = {}
if not os.path.exists(METADATA_FILE):
    print(f"!! Warning: metadata '{METADATA_FILE}' not found -> variable names used, marker 'o'.")
else:
    try:
        df_meta = pd.read_excel(METADATA_FILE, sheet_name=METADATA_SHEET)
        df_meta_map = df_meta.set_index('var').to_dict('index')
        print(f"[OK] Metadata loaded: {len(df_meta_map)} variables.")
    except Exception as e:
        print(f"!! Failed to load metadata: {e}")

# ===== Importance data =====
all_importances = None
if not os.path.exists(IMPORTANCE_FILE):
    print(f"!! Fatal error: '{IMPORTANCE_FILE}' not found.")
else:
    all_importances = pd.read_parquet(IMPORTANCE_FILE)
    _valid_years = sorted(pd.to_numeric(all_importances['year']).unique())
    print(f"[OK] Importance data loaded. Years: {_valid_years}")
    _missing = [y for y in SNAPSHOT_YEARS if int(y) not in _valid_years]
    if _missing:
        print(f"!! Warning: SNAPSHOT_YEARS not in the data: {_missing} -> those columns will be empty. Check the years.")


def prepare_ballot_ranks(all_importances, ballot, model, df_meta_map):
    """Filter ballot + model -> mean importance by year x feature -> rank, label and marker by year.
    Returns df_agg (year[str], feature, importance, rank[int], label, marker) or None if empty."""
    df = all_importances[(all_importances['ballot'] == ballot) & (all_importances['model'] == model)]
    df_agg = df.groupby(['year', 'feature'])['importance'].mean().reset_index()
    if df_agg.empty:
        return None
    df_agg['year'] = pd.to_numeric(df_agg['year']).astype(int)
    df_agg['rank'] = df_agg.groupby('year')['importance'].rank(method='first', ascending=False).astype(int)
    df_agg['year'] = df_agg['year'].astype(str)
    if df_meta_map:
        df_agg['label'] = df_agg['feature'].apply(lambda x: df_meta_map.get(x, {}).get('label', x))
        df_agg['marker'] = df_agg['feature'].apply(
            lambda x: 'D' if df_meta_map.get(x, {}).get('attitude', 2) == 1 else 'o')
    else:
        df_agg['label'] = df_agg['feature']
        df_agg['marker'] = 'o'
    return df_agg


def build_snapshot_plotdata(df_agg, snapshot_years, top_n, clipped_rank):
    """Union of the top-N variables of the snapshot years (key_features) and plotting data.
    categorize: the first snapshot year (in order) in which the variable is in the top N.
    Returns (plot_data, top_sets{year: set}, key_features)."""
    df_snap = df_agg[df_agg['year'].isin(snapshot_years)]
    top_sets = {yr: set(df_snap[(df_snap['year'] == yr) & (df_snap['rank'] <= top_n)]['feature'])
                for yr in snapshot_years}
    key_features = set().union(*top_sets.values()) if top_sets else set()
    plot_data = df_agg[df_agg['feature'].isin(key_features)].copy()
    plot_data['rank_clipped'] = plot_data['rank'].clip(upper=clipped_rank)
    plot_data['rank_display'] = plot_data['rank_clipped'].apply(
        lambda x: DISPLAY_RANK_SINK if x == clipped_rank else x)

    def categorize(feat):
        for yr in snapshot_years:
            if feat in top_sets[yr]:
                return yr
        return 'Other'

    plot_data['category'] = plot_data['feature'].apply(categorize)
    plot_data['year_num'] = pd.to_numeric(plot_data['year'])
    return plot_data, top_sets, key_features


# Snapshot year -> color (deep palette, in order)
_palette = sns.color_palette("deep", max(3, len(SNAPSHOT_YEARS)))
SNAPSHOT_COLORS = {yr: _palette[i] for i, yr in enumerate(SNAPSHOT_YEARS)}

print(f"[Config] SNAPSHOT_YEARS={SNAPSHOT_YEARS}, MODEL={TARGET_MODEL}, TOP_N={TOP_N}, suffix={YEAR_SUFFIX}")

### Table 1: top 10 variables in 1988, 2006 and 2024

In [ ]:
# === Table 1: top-N variables by rank (rows) x ballot and year (columns) ===
if all_importances is None:
    print("!! Importance data not loaded. Run the settings cell first.")
else:
    rows = []
    for target_ballot in ['a', 'b', 'c']:
        df_agg = prepare_ballot_ranks(all_importances, target_ballot, TARGET_MODEL, df_meta_map)
        if df_agg is None:
            print(f"-> No data for ballot '{target_ballot}'. Skipped."); continue
        df_agg['ballot'] = target_ballot
        rows.append(df_agg)

    if not rows:
        print("!! No data to aggregate.")
    else:
        combined = pd.concat(rows, ignore_index=True)
        sel = combined[(combined['year'].isin(SNAPSHOT_YEARS)) & (combined['rank'] <= TOP_N)].copy()
        sel['ballot_display'] = sel['ballot'].map({'a': 'Ballot A', 'b': 'Ballot B', 'c': 'Ballot C'})

        try:
            pivot = sel.pivot_table(index='rank', columns=['ballot_display', 'year'],
                                    values='label', aggfunc='first')
        except Exception as e:
            pivot = pd.DataFrame(); print(f"!! Pivot failed: {e}")

        if not pivot.empty:
            ballot_order = ['Ballot A', 'Ballot B', 'Ballot C']
            new_cols = pd.MultiIndex.from_product([ballot_order, SNAPSHOT_YEARS], names=['Ballot', 'Year'])
            pivot.columns = pivot.columns.rename(['Ballot', 'Year'])
            pivot = pivot.reindex(columns=new_cols)
            pivot = pivot.reindex(index=pd.Index(range(1, TOP_N + 1), name='Rank')).fillna('-')

            excel_filename = f'outputs/4_updown/4. ALL_BALLOTS_Top{TOP_N}_{TARGET_MODEL}_rank_as_rows_{YEAR_SUFFIX}.xlsx'
            pivot.to_excel(excel_filename)
            print(f"[Done] Table saved: '{excel_filename}'")
            print(f"\n--- Preview (ranks 1-{TOP_N}) ---")
            print(pivot)
        else:
            print("!! Pivot table could not be created; nothing saved.")

### Figure 2-B: mean number of attitude variables among the top 10 (mean over ballots)

In [ ]:
# === Number of attitude variables among the top N by year x ballot ===
def build_att_count_table(model, top_n=None):
    """Number of attitude variables among the top-N variables by year x ballot A/B/C for `model`.
    Returns DataFrame(index=Year[int], columns=['Ballot A','Ballot B','Ballot C']) or None."""
    if top_n is None:
        top_n = TOP_N
    counts = {}
    for tb in ['a', 'b', 'c']:
        df_agg = prepare_ballot_ranks(all_importances, tb, model, df_meta_map)
        if df_agg is None:
            continue
        top = df_agg[df_agg['rank'] <= top_n].copy()
        top['is_att'] = top['feature'].apply(
            lambda x: 1 if df_meta_map.get(x, {}).get('attitude', 2) == 1 else 0)
        top['year_int'] = pd.to_numeric(top['year']).astype(int)
        counts[f'Ballot {tb.upper()}'] = top.groupby('year_int')['is_att'].sum().sort_index()
    if not counts:
        return None
    tbl = pd.DataFrame(counts).sort_index()
    tbl.index.name = 'Year'
    return tbl


def assign_period(y, start0=1988, width=6, last_start=2018, last_end=2024):
    """Year -> 6-year period label 'start-end' starting in 1988.
    The last period is last_start-last_end (default 2018-2024), so 2024 is merged into it."""
    y = int(y)
    if y >= last_start:
        return f"{last_start}–{last_end}"
    start = start0 + width * ((y - start0) // width)
    return f"{start}–{start + width - 1}"


MODELS = ['XGBoost']
ballot_palette = sns.color_palette('deep', n_colors=3)
ballot_color = {'Ballot A': ballot_palette[0], 'Ballot B': ballot_palette[1], 'Ballot C': ballot_palette[2]}
ballot_marker = {'Ballot A': 'o', 'Ballot B': 's', 'Ballot C': '^'}
print("[OK] build_att_count_table / assign_period defined")

In [ ]:
# === Figure 2-B: mean over ballots A/B/C by year ===
if all_importances is None:
    print("!! Importance data not loaded. Run the settings cell first.")
else:
    for model in MODELS:
        tbl = build_att_count_table(model)
        if tbl is None:
            print(f"!! {model}: no data."); continue
        mean_ser = tbl.mean(axis=1).round(3)   # mean over the available ballots (skipna)

        out = tbl.copy()
        out['Mean (A,B,C)'] = mean_ser
        xlsx = f'outputs/4_updown/4. ALL_BALLOTS_Top{TOP_N}_{model}_attitude_MEAN_by_year.xlsx'
        out.to_excel(xlsx)

        # Same style as the performance figures
        # (points + line, title 16 / axes 12, no grid, x axis 1986-2026; no shading)
        png = f'outputs/4_updown/4. ALL_BALLOTS_Top{TOP_N}_{model}_attitude_MEAN_by_year.png'
        with plt.rc_context():
            plt.rcdefaults()
            plt.style.use('seaborn-v0_8-white')
            plt.figure(figsize=(12, 9))
            ax = plt.gca()
            color = sns.color_palette('deep', n_colors=1)[0]
            ax.plot(mean_ser.index, mean_ser.values, marker='o', linestyle='-', label='Mean (A, B, C)', color=color)

            ax.set_title(f'Mean Number of Attitude Variables in Top {TOP_N} by Year', fontsize=16, pad=20)
            ax.set_xlabel('Year', fontsize=12)
            ax.set_ylabel(f'Attitude count (max {TOP_N})', fontsize=12)
            ax.legend()
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_ylim(0, TOP_N)
            ax.set_xlim(1986, 2026)

            plt.tight_layout()
            plt.savefig(png, dpi=300)
            plt.close()
        print(f"[Done] {model}: '{png}' + '{xlsx}'")

## Figure 2-C: without attitude variables

In [ ]:
### Performance by year

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Settings ---
# Result file
excel_file_path = 'outputs/4_parsimonious/4_1_Final_Att_Excluded_partyid_1_analysis_results_weighted.xlsx'

# Output folder
output_directory = 'outputs/4_parsimonious/plots'

# Sheet
sheet_name = '2_Performance_Compare'

# --- Read data ---
try:
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
    print(f"Read sheet '{sheet_name}' of '{excel_file_path}'.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: '{excel_file_path}' not found.")
except ValueError:
    print(f"Error: sheet '{sheet_name}' not found in '{excel_file_path}'.")
except Exception as e:
    print(f"Error while reading the Excel file: {e}")
    df = None # skip the plotting below on error

# --- Prepare data and plot ---
if df is not None:
    # Check required columns
    required_cols = ['year', 'ballot', 'model', 'avg_score_mean', 'avg_score_std']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: required columns ({required_cols}) missing.")
    else:
        # 'year' to numeric
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        # Keep rows with a valid 'year'
        df.dropna(subset=['year'], inplace=True)
        df['year'] = df['year'].astype(int)

        # Create the output folder if needed
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)

        # Plot style
        plt.style.use('seaborn-v0_8-white')

        # One plot per model
        models_to_process = df['model'].unique()

        print("\nPlotting...")

        for model_name in models_to_process:
            model_data = df[df['model'] == model_name].copy()

            if model_data.empty:
                print(f"No data for model '{model_name}'; no plot.")
                continue

            plt.figure(figsize=(12,9))
            ax = plt.gca()

            # Ballots in model_data
            ballots = model_data['ballot'].unique()

            # One color per ballot (seaborn deep palette)
            palette = sns.color_palette('deep', n_colors=len(ballots))

            for i, ballot_name in enumerate(ballots):
                # Data of one ballot, sorted by year
                ballot_data = model_data[model_data['ballot'] == ballot_name].sort_values('year')

                if ballot_data.empty:
                    continue

                x = ballot_data['year']
                y_mean = ballot_data['avg_score_mean']
                y_std = ballot_data['avg_score_std']

                # Mean score (avg_score_mean) as a line
                ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])

                # Standard deviation (avg_score_std) as a shaded band
                # Shade: ±2 SD across the 20 repetitions
                ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)



            # Title and axis labels
            ax.set_title(f'Final Model, WITHOUT Attitude Variables', fontsize=16, pad=20)
            ax.set_xlabel('Year', fontsize=12)
            ax.set_ylabel('Average Score (AUROC)', fontsize=12)
            ax.legend(title='Ballot')

            # No grid
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_ylim(0.5, 1.0)
            ax.set_xlim(1986, 2026)

            plt.tight_layout()

            # Save
            filename = f'{model_name}_Final_NO_Attitude_performance_over_time.png'
            filepath = os.path.join(output_directory, filename)
            plt.savefig(filepath, dpi=300)
            print(f"Saved '{filepath}'.")

            plt.close()

        print("\nAll plots done.")

## Figure 2-D: attitude variables only

In [ ]:
### Performance by year

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Settings ---
# Result file
excel_file_path = 'outputs/4_parsimonious/4_1_Final_Att_Only_partyid_1_analysis_results_weighted.xlsx'

# Output folder
output_directory = 'outputs/4_parsimonious/plots'

# Sheet
sheet_name = '2_Performance_Compare'

# --- Read data ---
try:
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
    print(f"Read sheet '{sheet_name}' of '{excel_file_path}'.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: '{excel_file_path}' not found.")
except ValueError:
    print(f"Error: sheet '{sheet_name}' not found in '{excel_file_path}'.")
except Exception as e:
    print(f"Error while reading the Excel file: {e}")
    df = None # skip the plotting below on error

# --- Prepare data and plot ---
if df is not None:
    # Check required columns
    required_cols = ['year', 'ballot', 'model', 'avg_score_mean', 'avg_score_std']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: required columns ({required_cols}) missing.")
    else:
        # 'year' to numeric
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        # Keep rows with a valid 'year'
        df.dropna(subset=['year'], inplace=True)
        df['year'] = df['year'].astype(int)

        # Create the output folder if needed
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)

        # Plot style
        plt.style.use('seaborn-v0_8-white')

        # One plot per model
        models_to_process = df['model'].unique()

        print("\nPlotting...")

        for model_name in models_to_process:
            model_data = df[df['model'] == model_name].copy()

            if model_data.empty:
                print(f"No data for model '{model_name}'; no plot.")
                continue

            plt.figure(figsize=(12,9))
            ax = plt.gca()

            # Ballots in model_data
            ballots = model_data['ballot'].unique()

            # One color per ballot (seaborn deep palette)
            palette = sns.color_palette('deep', n_colors=len(ballots))

            for i, ballot_name in enumerate(ballots):
                # Data of one ballot, sorted by year
                ballot_data = model_data[model_data['ballot'] == ballot_name].sort_values('year')

                if ballot_data.empty:
                    continue

                x = ballot_data['year']
                y_mean = ballot_data['avg_score_mean']
                y_std = ballot_data['avg_score_std']

                # Mean score (avg_score_mean) as a line
                ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])

                # Standard deviation (avg_score_std) as a shaded band
                # Shade: ±2 SD across the 20 repetitions
                ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)



            # Title and axis labels
            ax.set_title(f'Final Model, ONLY WITH Attitude Variables', fontsize=16, pad=20)
            ax.set_xlabel('Year', fontsize=12)
            ax.set_ylabel('Average Score (AUROC)', fontsize=12)
            ax.legend(title='Ballot')

            # No grid
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_ylim(0.5, 1.0)
            ax.set_xlim(1986, 2026)

            plt.tight_layout()

            # Save
            filename = f'{model_name}_Final_Attitude_performance_over_time.png'
            filepath = os.path.join(output_directory, filename)
            plt.savefig(filepath, dpi=300)
            print(f"Saved '{filepath}'.")

            plt.close()

        print("\nAll plots done.")

## Figure 2-E: models by variable category

In [ ]:
### Prediction performance by variable category (Attitude / Demography / SES / Lifestyle + Social Capital)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

RESULT_DIR = 'outputs/4_parsimonious'
category_files = {
    # The Attitude model has the same variable set as 4_1_Final_Att_Only, so its results are used
    'Attitude':                     f'{RESULT_DIR}/4_1_Final_Att_Only_partyid_1_analysis_results_weighted.xlsx',
    'Demography':                   f'{RESULT_DIR}/4_1_Cat_Demography_partyid_1_analysis_results_weighted.xlsx',
    'SES':                          f'{RESULT_DIR}/4_1_Cat_SES_partyid_1_analysis_results_weighted.xlsx',
    'Lifestyle + Social Capital':   f'{RESULT_DIR}/4_1_Cat_LifestyleSoccap_partyid_1_analysis_results_weighted.xlsx',
}
reference_file = 'outputs/3_final/3_FINAL_M3_Top40_partyid_1_analysis_results_weighted.xlsx'  # reference line: final model
output_directory = f'{RESULT_DIR}/plots'
os.makedirs(output_directory, exist_ok=True)


def load_auc(path):
    """From the scores parquet (one row per run), take roc_auc_ovr, average over ballots per run and (year, model),
    then return the mean and SD over the 20 runs. path is the result xlsx (the parquet with the same name is read)."""
    pq = path.replace('analysis_results_weighted.xlsx', 'analysis_all_scores_weighted.parquet')
    df = pd.read_parquet(pq)
    df = df[df['metric'] == 'roc_auc_ovr'].copy()
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype(int)
    per_run = df.groupby(['year', 'model', 'run'])['avg_score'].mean().reset_index()
    agg = per_run.groupby(['year', 'model'])['avg_score'].agg(['mean', 'std']).reset_index()
    agg.columns = ['year', 'model', 'avg_score_mean', 'avg_score_std']
    return agg

all_data = {label: load_auc(path) for label, path in category_files.items()}
reference = load_auc(reference_file)

plt.style.use('seaborn-v0_8-white')
palette = sns.color_palette('deep', n_colors=len(category_files))

for model_name in ['XGBoost']:
    plt.figure(figsize=(12, 9))
    ax = plt.gca()

    ref = reference[reference['model'] == model_name].sort_values('year')
    ax.plot(ref['year'], ref['avg_score_mean'], linestyle='--', color=sns.color_palette('deep')[7], label='Final (all 40 features)')

    for (label, agg), color in zip(all_data.items(), palette):
        model_data = agg[agg['model'] == model_name].sort_values('year')
        if model_data.empty:
            continue
        x = model_data['year']
        y_mean = model_data['avg_score_mean']
        y_std = model_data['avg_score_std'].fillna(0)
        ax.plot(x, y_mean, marker='o', linestyle='-', label=label, color=color)
        # Shade: ±2 SD across the 20 repetitions
        ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=color, alpha=0.2)

    ax.set_title('Prediction Performance by Variable Category', fontsize=16, pad=20)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Average Score (AUROC)', fontsize=12)
    ax.legend()
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0.5, 1.0)
    ax.set_xlim(1986, 2026)

    plt.tight_layout()
    filepath = os.path.join(output_directory, f'{model_name}_comparison_category_models.png')
    plt.savefig(filepath, dpi=300)
    print(f'Saved: {filepath}')
    plt.close()

# Mean AUROC by period (over ballots and years)
PERIODS = {'1988-1999': (1988, 1999), '2000-2009': (2000, 2009), '2010-2018': (2010, 2018),
           '2021-2024': (2021, 2024), 'All': (1988, 2024)}
rows = []
for label, agg in {**all_data, 'Final (all 40 features)': reference}.items():
    for model_name in ['XGBoost']:
        d = agg[agg['model'] == model_name]
        row = {'feature_set': label, 'model': model_name}
        for p, (lo, hi) in PERIODS.items():
            row[p] = d[(d['year'] >= lo) & (d['year'] <= hi)]['avg_score_mean'].mean()
        rows.append(row)
summary = pd.DataFrame(rows)
summary_path = f'{RESULT_DIR}/4_1_Cat_comparison_summary.xlsx'
summary.to_excel(summary_path, index=False)
print(summary.round(3).to_string(index=False))
print(f'Saved: {summary_path}')

## Figure 2-F: top 3 variables of each ballot after 2020

In [ ]:
### Performance by year

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Settings ---
# Result file
excel_file_path = 'outputs/4_parsimonious/4_1_2020s_Parsimon_top3_only_partyid_1_analysis_results_weighted.xlsx'

# Output folder
output_directory = 'outputs/4_parsimonious/plots'

# Sheet
sheet_name = '2_Performance_Compare'

# --- Read data ---
try:
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
    print(f"Read sheet '{sheet_name}' of '{excel_file_path}'.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: '{excel_file_path}' not found.")
except ValueError:
    print(f"Error: sheet '{sheet_name}' not found in '{excel_file_path}'.")
except Exception as e:
    print(f"Error while reading the Excel file: {e}")
    df = None # skip the plotting below on error

# --- Prepare data and plot ---
if df is not None:
    # Check required columns
    required_cols = ['year', 'ballot', 'model', 'avg_score_mean', 'avg_score_std']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: required columns ({required_cols}) missing.")
    else:
        # 'year' to numeric
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        # Keep rows with a valid 'year'
        df.dropna(subset=['year'], inplace=True)
        df['year'] = df['year'].astype(int)

        # Create the output folder if needed
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)

        # Plot style
        plt.style.use('seaborn-v0_8-white')

        # One plot per model
        models_to_process = df['model'].unique()

        print("\nPlotting...")

        for model_name in models_to_process:
            model_data = df[df['model'] == model_name].copy()

            if model_data.empty:
                print(f"No data for model '{model_name}'; no plot.")
                continue

            plt.figure(figsize=(12,9))
            ax = plt.gca()

            # Ballots in model_data
            ballots = model_data['ballot'].unique()

            # One color per ballot (seaborn deep palette)
            palette = sns.color_palette('deep', n_colors=len(ballots))

            for i, ballot_name in enumerate(ballots):
                # Data of one ballot, sorted by year
                ballot_data = model_data[model_data['ballot'] == ballot_name].sort_values('year')

                if ballot_data.empty:
                    continue

                x = ballot_data['year']
                y_mean = ballot_data['avg_score_mean']
                y_std = ballot_data['avg_score_std']

                # Mean score (avg_score_mean) as a line
                ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])

                # Standard deviation (avg_score_std) as a shaded band
                # Shade: ±2 SD across the 20 repetitions
                ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)



            # Title and axis labels
            ax.set_title(f"Reduced Model: Each Ballot's Top 3 Variables After 2020", fontsize=16, pad=20)
            ax.set_xlabel('Year', fontsize=12)
            ax.set_ylabel('Average Score (AUROC)', fontsize=12)
            ax.legend(title='Ballot')

            # No grid
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_ylim(0.5, 1.0)
            ax.set_xlim(1986, 2026)

            plt.tight_layout()

            # Save
            filename = f'{model_name}_Reduced_Top3_Attitude_performance_over_time.png'
            filepath = os.path.join(output_directory, filename)
            plt.savefig(filepath, dpi=300)
            print(f"Saved '{filepath}'.")

            plt.close()

        print("\nAll plots done.")

## Figure S4: single variable (number of immigrants)

In [ ]:
### Performance by year

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Settings ---
# Result file
excel_file_path = 'outputs/4_parsimonious/4_1_Single_letin1a_partyid_1_analysis_results_weighted.xlsx'

# Output folder
output_directory = 'outputs/4_parsimonious/plots'

# Sheet
sheet_name = '2_Performance_Compare'

# --- Read data ---
try:
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
    print(f"Read sheet '{sheet_name}' of '{excel_file_path}'.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: '{excel_file_path}' not found.")
except ValueError:
    print(f"Error: sheet '{sheet_name}' not found in '{excel_file_path}'.")
except Exception as e:
    print(f"Error while reading the Excel file: {e}")
    df = None # skip the plotting below on error

# --- Prepare data and plot ---
if df is not None:
    # Check required columns
    required_cols = ['year', 'ballot', 'model', 'avg_score_mean', 'avg_score_std']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: required columns ({required_cols}) missing.")
    else:
        # 'year' to numeric
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        # Keep rows with a valid 'year'
        df.dropna(subset=['year'], inplace=True)
        df['year'] = df['year'].astype(int)

        # Create the output folder if needed
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)

        # Plot style
        plt.style.use('seaborn-v0_8-white')

        # One plot per model
        models_to_process = df['model'].unique()

        print("\nPlotting...")

        for model_name in models_to_process:
            model_data = df[df['model'] == model_name].copy()

            if model_data.empty:
                print(f"No data for model '{model_name}'; no plot.")
                continue

            plt.figure(figsize=(12,9))
            ax = plt.gca()

            # Ballots in model_data
            ballots = model_data['ballot'].unique()

            # One color per ballot (seaborn deep palette)
            palette = sns.color_palette('deep', n_colors=len(ballots))

            for i, ballot_name in enumerate(ballots):
                # Data of one ballot, sorted by year
                ballot_data = model_data[model_data['ballot'] == ballot_name].sort_values('year')

                if ballot_data.empty:
                    continue

                x = ballot_data['year']
                y_mean = ballot_data['avg_score_mean']
                y_std = ballot_data['avg_score_std']

                # Mean score (avg_score_mean) as a line
                ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])

                # Standard deviation (avg_score_std) as a shaded band
                # Shade: ±2 SD across the 20 repetitions
                ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)



            # Title and axis labels
            ax.set_title(f'Single Feature: N of Immigrants Should Be?', fontsize=16, pad=20)
            ax.set_xlabel('Year', fontsize=12)
            ax.set_ylabel('Average Score (AUROC)', fontsize=12)
            ax.legend(title='Ballot')

            # No grid
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_ylim(0.5, 1.0)
            ax.set_xlim(1986, 2026)

            plt.tight_layout()

            # Save
            filename = f'{model_name}_Single_letin1a_performance_over_time.png'
            filepath = os.path.join(output_directory, filename)
            plt.savefig(filepath, dpi=300)
            print(f"Saved '{filepath}'.")

            plt.close()

        print("\nAll plots done.")

## Figure 5: race and sex models

In [ ]:
### Performance by year (race_binary / sex)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Settings ---
# Targets
TARGETS = ['race_binary', 'sex']

# Output folder
output_directory = 'outputs/7_final_race_sex/plots'

# Sheet
sheet_name = '2_Performance_Compare'

# Create the output folder if needed
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Plot style
plt.style.use('seaborn-v0_8-white')

for target in TARGETS:
    # Result file
    excel_file_path = f'outputs/7_final_race_sex/7_FINAL_M3_Top40_{target}_analysis_results_weighted.xlsx'

    # --- Read data ---
    df = None
    try:
        df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
        print(f"Read sheet '{sheet_name}' of '{excel_file_path}'.")
        display(df.head())
    except FileNotFoundError:
        print(f"Error: '{excel_file_path}' not found.")
    except ValueError:
        print(f"Error: sheet '{sheet_name}' not found in '{excel_file_path}'.")
    except Exception as e:
        print(f"Error while reading the Excel file: {e}")
        df = None  # skip the plotting below on error

    # --- Prepare data and plot ---
    if df is not None:
        # Check required columns
        required_cols = ['year', 'ballot', 'model', 'avg_score_mean', 'avg_score_std']
        if not all(col in df.columns for col in required_cols):
            print(f"Error: required columns ({required_cols}) missing.")
        else:
            # 'year' to numeric
            df['year'] = pd.to_numeric(df['year'], errors='coerce')
            # Keep rows with a valid 'year'
            df.dropna(subset=['year'], inplace=True)
            df['year'] = df['year'].astype(int)

            # One plot per model
            models_to_process = df['model'].unique()

            print(f"\n[{target}] Plotting...")

            for model_name in models_to_process:
                model_data = df[df['model'] == model_name].copy()

                if model_data.empty:
                    print(f"No data for model '{model_name}'; no plot.")
                    continue

                plt.figure(figsize=(12, 9))
                ax = plt.gca()

                # Ballots in model_data
                ballots = model_data['ballot'].unique()

                # One color per ballot (seaborn deep palette)
                palette = sns.color_palette('deep', n_colors=len(ballots))

                for i, ballot_name in enumerate(ballots):
                    # Data of one ballot, sorted by year
                    ballot_data = model_data[model_data['ballot'] == ballot_name].sort_values('year')

                    if ballot_data.empty:
                        continue

                    x = ballot_data['year']
                    y_mean = ballot_data['avg_score_mean']
                    y_std = ballot_data['avg_score_std']

                    # Mean score (avg_score_mean) as a line
                    ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])

                    # Standard deviation (avg_score_std) as a shaded band
                    # fill_between fills the range of y for each x.
                    ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)


                # Title and axis labels
                ax.set_title(f'Final Model Performance ({target})', fontsize=16, pad=20)
                ax.set_xlabel('Year', fontsize=12)
                ax.set_ylabel('Average Score (AUROC)', fontsize=12)
                ax.legend(title='Ballot')
                ax.set_ylim(0.5, 1)
                ax.set_xlim(1986, 2026)

                # No grid
                ax.grid(False)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)

                plt.tight_layout()

                # Save
                filename = f'{target}_{model_name}_final_model.png'
                filepath = os.path.join(output_directory, filename)
                plt.savefig(filepath, dpi=300)
                print(f"Saved '{filepath}'.")

                plt.close()

    print(f"[{target}] All plots done.\n")

## Figure S1-A: preliminary model

In [ ]:
### Performance by year

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Settings ---
# Result file
excel_file_path = 'outputs/1_preliminary/1_Preliminary_partyid_1_analysis_results_weighted.xlsx'

# Output folder
output_directory = 'outputs/1_preliminary/plots'

# Sheet
sheet_name = '2_Performance_Compare'

# --- Read data ---
try:
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
    print(f"Read sheet '{sheet_name}' of '{excel_file_path}'.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: '{excel_file_path}' not found.")
except ValueError:
    print(f"Error: sheet '{sheet_name}' not found in '{excel_file_path}'.")
except Exception as e:
    print(f"Error while reading the Excel file: {e}")
    df = None # skip the plotting below on error

# --- Prepare data and plot ---
if df is not None:
    # Check required columns
    required_cols = ['year', 'ballot', 'model', 'avg_score_mean', 'avg_score_std']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: required columns ({required_cols}) missing.")
    else:
        # 'year' to numeric
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        # Keep rows with a valid 'year'
        df.dropna(subset=['year'], inplace=True)
        df['year'] = df['year'].astype(int)

        # Create the output folder if needed
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)

        # Plot style
        plt.style.use('seaborn-v0_8-white')

        # One plot per model
        models_to_process = df['model'].unique()

        print("\nPlotting...")

        for model_name in models_to_process:
            model_data = df[df['model'] == model_name].copy()

            if model_data.empty:
                print(f"No data for model '{model_name}'; no plot.")
                continue

            plt.figure(figsize=(12, 9))
            ax = plt.gca()

            # Ballots in model_data
            ballots = model_data['ballot'].unique()

            # One color per ballot (seaborn deep palette)
            palette = sns.color_palette('deep', n_colors=len(ballots))

            for i, ballot_name in enumerate(ballots):
                # Data of one ballot, sorted by year
                ballot_data = model_data[model_data['ballot'] == ballot_name].sort_values('year')

                if ballot_data.empty:
                    continue

                x = ballot_data['year']
                y_mean = ballot_data['avg_score_mean']
                y_std = ballot_data['avg_score_std']

                # Mean score (avg_score_mean) as a line
                ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])

                # Standard deviation (avg_score_std) as a shaded band
                # Shade: ±2 SD across the 20 repetitions
                ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)



            # Title and axis labels
            ax.set_title(f'Preliminary Model Performance Over Time', fontsize=16, pad=20)
            ax.set_xlabel('Year', fontsize=12)
            ax.set_ylabel('Average Score (AUROC)', fontsize=12)
            ax.legend(title='Ballot')
            ax.set_ylim(0.5,1)
            ax.set_xlim(1986, 2025)

            # No grid
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            plt.tight_layout()

            # Save
            filename = f'{model_name}_performance_over_time.png'
            filepath = os.path.join(output_directory, filename)
            plt.savefig(filepath, dpi=300)
            print(f"Saved '{filepath}'.")

            plt.close()

        print("\nAll plots done.")

## Figure S2: preliminary logistic regression

In [ ]:
### Preliminary logistic regression: performance by year

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Settings ---
# Result file
LOGIT_FILES = {
    'Logit_OneHot':  'outputs/1_1_preliminary_logit/1_Logit_OneHot_partyid_1_analysis_results_weighted.xlsx',
    'Logit_Numeric': 'outputs/1_1_preliminary_logit/1_Logit_Numeric_partyid_1_analysis_results_weighted.xlsx',
}
LOGIT_TITLES = {
    'Logit_OneHot':  'Preliminary Logistic Regression Performance Over Time (One-Hot Encoding)',
    'Logit_Numeric': 'Preliminary Logistic Regression Performance Over Time (Numeric Encoding)',
}

# Output folder
output_directory = 'outputs/1_1_preliminary_logit/plots'

# Sheet
sheet_name = '2_Performance_Compare'

for encoding, excel_file_path in LOGIT_FILES.items():
    # --- Read data ---
    try:
        df = pd.read_excel(excel_file_path, sheet_name=sheet_name)
        print(f"Read sheet '{sheet_name}' of '{excel_file_path}'.")
        display(df.head())
    except FileNotFoundError:
        print(f"Error: '{excel_file_path}' not found.")
    except ValueError:
        print(f"Error: sheet '{sheet_name}' not found in '{excel_file_path}'.")
    except Exception as e:
        print(f"Error while reading the Excel file: {e}")
        df = None # skip the plotting below on error

    # --- Prepare data and plot ---
    if df is not None:
        # Check required columns
        required_cols = ['year', 'ballot', 'model', 'avg_score_mean', 'avg_score_std']
        if not all(col in df.columns for col in required_cols):
            print(f"Error: required columns ({required_cols}) missing.")
        else:
            # 'year' to numeric
            df['year'] = pd.to_numeric(df['year'], errors='coerce')
            # Keep rows with a valid 'year'
            df.dropna(subset=['year'], inplace=True)
            df['year'] = df['year'].astype(int)

            # Create the output folder if needed
            if not os.path.exists(output_directory):
                os.makedirs(output_directory)

            # Plot style
            plt.style.use('seaborn-v0_8-white')

            # One plot per model
            models_to_process = df['model'].unique()

            print("\nPlotting...")

            for model_name in models_to_process:
                model_data = df[df['model'] == model_name].copy()

                if model_data.empty:
                    print(f"No data for model '{model_name}'; no plot.")
                    continue

                plt.figure(figsize=(12, 9))
                ax = plt.gca()

                # Ballots in model_data
                ballots = model_data['ballot'].unique()

                # One color per ballot (seaborn deep palette)
                palette = sns.color_palette('deep', n_colors=len(ballots))

                for i, ballot_name in enumerate(ballots):
                    # Data of one ballot, sorted by year
                    ballot_data = model_data[model_data['ballot'] == ballot_name].sort_values('year')

                    if ballot_data.empty:
                        continue

                    x = ballot_data['year']
                    y_mean = ballot_data['avg_score_mean']
                    y_std = ballot_data['avg_score_std']

                    # Mean score (avg_score_mean) as a line
                    ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])

                    # Standard deviation (avg_score_std) as a shaded band
                    # Shade: ±2 SD across the 20 repetitions
                    ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)



                # Title and axis labels
                ax.set_title(LOGIT_TITLES[encoding], fontsize=16, pad=20)
                ax.set_xlabel('Year', fontsize=12)
                ax.set_ylabel('Average Score (AUROC)', fontsize=12)
                ax.legend(title='Ballot')
                ax.set_ylim(0.5,1)
                ax.set_xlim(1986, 2025)

                # No grid
                ax.grid(False)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)

                plt.tight_layout()

                # Save
                filename = f'{encoding}_performance_over_time.png'
                filepath = os.path.join(output_directory, filename)
                plt.savefig(filepath, dpi=300)
                print(f"Saved '{filepath}'.")

                plt.close()

            print("\nAll plots done.")

## Figure S3: AUROC by number of selected variables and selection criterion

In [ ]:
# All three ballots in one plot (one plot each for M1 / M3 / M4)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# --- Settings ---
# {method} and {top_n} are filled in with the selection criterion and N.
base_filename = 'outputs/2_feature_selection/2_Feature Selection_{method}_Top{top_n}_partyid_1_analysis_all_scores_weighted.parquet'
top_values = [10, 20, 30, 40, 50, 60, 70, 80, 90]
output_directory = 'outputs/2_feature_selection/plots/'
os.makedirs(output_directory, exist_ok=True)

# Plot titles by selection criterion (based on the SHAP importance of the preliminary model)
METHOD_TITLES = {
    'M1': 'Features Ranked by Mean SHAP Importance',
    'M3': 'Features Ranked by Number of Times in the Top 20',
    'M4': 'Features Ranked by Normalized Frequency in the Top 20',
}

for method, method_title in METHOD_TITLES.items():
    print(f"\n--- {method}: performance by N ---")
    rows = []
    for top_n in top_values:
        parquet_file_path = base_filename.format(method=method, top_n=top_n)
        # Stop if a result file is missing, so the plot is not silently incomplete.
        if not os.path.exists(parquet_file_path):
            raise FileNotFoundError(f"'{parquet_file_path}' not found.")
        df_raw_scores = pd.read_parquet(parquet_file_path)
        df_raw_scores = df_raw_scores[(df_raw_scores['model'] == 'XGBoost') & (df_raw_scores['metric'] == 'roc_auc_ovr')]
        # Per run: mean AUROC over all years of the ballot -> mean/SD over runs (same basis as the shades of the performance figures)
        per_run = df_raw_scores.groupby(['ballot', 'run'])['avg_score'].mean().reset_index()
        summary = per_run.groupby('ballot')['avg_score'].agg(['mean', 'std']).reset_index()
        summary['Top_N'] = top_n
        rows.append(summary)
    perf = pd.concat(rows, ignore_index=True).rename(columns={'mean': 'avg_score_mean', 'std': 'avg_score_std'})
    print(perf.pivot(index='Top_N', columns='ballot', values='avg_score_mean').round(3))

    plt.style.use('seaborn-v0_8-white')
    plt.figure(figsize=(12, 9))
    ax = plt.gca()
    ballots = sorted(perf['ballot'].unique())
    palette = sns.color_palette('deep', n_colors=len(ballots))

    for i, ballot_name in enumerate(ballots):
        ballot_data = perf[perf['ballot'] == ballot_name].sort_values('Top_N')
        x = ballot_data['Top_N']
        y_mean = ballot_data['avg_score_mean']
        y_std = ballot_data['avg_score_std'].fillna(0)

        ax.plot(x, y_mean, marker='o', linestyle='-', label=ballot_name, color=palette[i])
        # Shade: ±2 SD across the 20 repetitions
        ax.fill_between(x, y_mean - 2 * y_std, y_mean + 2 * y_std, color=palette[i], alpha=0.2)

    ax.set_title(f'Average AUROC by Number of Features\n({method_title})', fontsize=16, pad=20)
    ax.set_xlabel('Number of Top N Features', fontsize=12)
    ax.set_ylabel('Average Score (AUROC)', fontsize=12)
    ax.legend(title='Ballot')
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0.5, 1.0)
    ax.set_xticks(top_values)

    plt.tight_layout()
    filepath = os.path.join(output_directory, f'{method}_AllBallots_Performance_by_TopN.png')
    plt.savefig(filepath, dpi=300)
    print(f"Saved '{filepath}'.")
    plt.close()